In [ ]:
import logging
import os
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, ArrayType
from pyspark.sql.functions import (
    col, split, explode, when, lower, upper, lit, size, array, struct,
    row_number, desc, avg, input_file_name, broadcast, trim, count, max, sum, countDistinct, collect_set, first
)
from pyspark import StorageLevel

# ==========================================
# CẤU HÌNH HỆ THỐNG COLAB & SPARK
# ==========================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("Colab_Biomedical_ETL")

# Tối ưu hóa RAM cho Google Colab (12GB RAM)
spark = SparkSession.builder \
    .appName("GDC_GEO_STRING_Colab_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "30") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.2") \
    .getOrCreate()

# Đường dẫn Google Drive Colab
BASE_DIR = "/content/drive/MyDrive/drugtarget_local"
RAW_DIR = f"{BASE_DIR}/raw"
REFINED_DIR = f"{BASE_DIR}/refined"
numeric_regex = r"^[+-]?([0-9]*[.])?[0-9]+([eE][+-]?[0-9]+)?$"

logger.info("======= BẮT ĐẦU CHẠY PIPELINE ETL GIAI ĐOẠN 1 (GDC + GEO + STRING) TRÊN COLAB =======")

# ==========================================
# BƯỚC 1: XỬ LÝ GDC COUNTS
# ==========================================
logger.info("[Bước 1] GDC Counts...")
gdc_counts_schema = StructType([
    StructField("gene_id", StringType(), True), StructField("gene_name", StringType(), True),
    StructField("gene_type", StringType(), True), StructField("unstranded", LongType(), True),
    StructField("stranded_first", LongType(), True), StructField("stranded_second", LongType(), True),
    StructField("tpm_unstranded", StringType(), True), StructField("fpkm_unstranded", StringType(), True),
    StructField("fpkm_uq_unstranded", StringType(), True)
])

df_counts_raw = spark.read.option("header", "true").option("sep", "\t").option("comment", "#") \
    .schema(gdc_counts_schema).csv(f"{RAW_DIR}/gdc/counts/*/*.tsv")

df_counts_raw = df_counts_raw.withColumn("file_id", split(input_file_name(), "/").getItem(size(split(input_file_name(), "/")) - 2))

for c in ["tpm_unstranded", "fpkm_unstranded", "fpkm_uq_unstranded"]:
    df_counts_raw = df_counts_raw.withColumn(c, when(col(c).rlike(numeric_regex), col(c).cast(DoubleType())).otherwise(lit(0.0)))

df_counts_clean = df_counts_raw.withColumn("gene_id", split(col("gene_id"), r"\.")[0])
window_spec = Window.partitionBy("file_id", "gene_id").orderBy(desc("tpm_unstranded"))
df_counts_clean = df_counts_clean.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")
df_counts_clean = df_counts_clean.filter((~col("gene_id").startswith("N_")) & (col("tpm_unstranded") >= 0.1) & (col("gene_type") == "protein_coding"))

df_counts_clean.persist(StorageLevel.MEMORY_AND_DISK)
df_counts_clean.repartition(8).write.mode("overwrite").parquet(f"{REFINED_DIR}/gdc_counts_clean.parquet")


# ==========================================
# BƯỚC 2: XỬ LÝ GDC SAMPLE_CASE (VÁ LỖI CỘT BỊ FLATTEN)
# ==========================================
logger.info("[Bước 2] GDC Sample Case...")
gdc_index_schema = StructType([StructField("data", StructType([StructField("hits", ArrayType(StructType([
    StructField("file_id", StringType(), True), StructField("cases", ArrayType(StructType([StructField("case_id", StringType(), True)])), True)
])), True)]), True)])

df_index_flat = spark.read.option("multiline", "true").schema(gdc_index_schema).json(f"{RAW_DIR}/gdc/metadata/files_index/*/*.json") \
    .select(explode(col("data.hits")).alias("hit")).select(col("hit.file_id").alias("file_id"), explode(col("hit.cases")).alias("case_node")) \
    .select("file_id", col("case_node.case_id").alias("case_id"))

df_final_sample_case = spark.read.option("header", "true").option("sep", "\t").csv(f"{RAW_DIR}/gdc/metadata/cases_samples/*/*.tsv")

# [VÁ LỖI TẠI ĐÂY]: Ánh xạ các cột có đường dẫn lồng nhau về tên chuẩn
column_mapping = {
    "project.project_id": "project_id",
    "demographic.gender": "gender",
    "demographic.race": "race",
    "demographic.days_to_death": "days_to_death",
    "diagnoses.0.days_to_last_follow_up": "days_to_last_follow_up",
    "diagnoses.0.primary_diagnosis": "primary_diagnosis",
    "diagnoses.0.tissue_or_organ_of_origin": "tissue_or_organ_of_origin",
    "samples.0.sample_type": "sample_type"
}

for old_col, new_col in column_mapping.items():
    df_final_sample_case = df_final_sample_case.withColumnRenamed(old_col, new_col)

# Đảm bảo các cột cần thiết chắc chắn tồn tại (nếu file TSV thiếu cột)
required_cols = ["days_to_death", "days_to_last_follow_up", "sample_type", "primary_diagnosis", "tissue_or_organ_of_origin"]
for c in required_cols:
    if c not in df_final_sample_case.columns:
        df_final_sample_case = df_final_sample_case.withColumn(c, lit(None))

# Ép kiểu an toàn cho các cột ngày tháng
for c in ["days_to_death", "days_to_last_follow_up"]:
    df_final_sample_case = df_final_sample_case.withColumn(c, col(c).cast(DoubleType()))

# Logic gán Tumor/Normal
df_final_sample_case = df_final_sample_case.withColumn("raw_sample_type", lower(col("sample_type")))
df_final_sample_case = df_final_sample_case.withColumn("sample_type",
    when(col("raw_sample_type").contains("tumor") | col("raw_sample_type").contains("cancer") | col("raw_sample_type").contains("metastatic"), "tumor")
    .when(col("raw_sample_type").contains("normal"), "normal").otherwise("other")
).drop("raw_sample_type")

# Logic sinh tồn
df_final_sample_case = df_final_sample_case.withColumn("event_observed", when(col("days_to_death").isNotNull() & (col("days_to_death") > 0), 1).otherwise(0))
df_final_sample_case = df_final_sample_case.withColumn(
    "survival_time", when(col("event_observed") == 1, col("days_to_death")).otherwise(when(col("days_to_last_follow_up").isNotNull(), col("days_to_last_follow_up")).otherwise(-1.0))
)

df_final_sample_case = df_final_sample_case.filter(col("sample_type") != "other")
df_gdc_sample_case_joined = df_final_sample_case.join(df_index_flat, on="case_id", how="left").dropDuplicates(["case_id", "sample_type"])

df_gdc_sample_case_joined.persist(StorageLevel.MEMORY_AND_DISK)
df_gdc_sample_case_joined.write.mode("overwrite").partitionBy("sample_type").parquet(f"{REFINED_DIR}/gdc_sample_case_clean.parquet")


# ==========================================
# BƯỚC 3: GDC ANNOTATE (BROADCAST JOIN)
# ==========================================
logger.info("[Bước 3] GDC Annotate...")
df_gdc_annotate = df_counts_clean.join(broadcast(df_gdc_sample_case_joined), on="file_id", how="inner")
df_gdc_annotate.repartition(16).write.mode("overwrite").parquet(f"{REFINED_DIR}/gdc/annotate.parquet")

df_counts_clean.unpersist()
df_gdc_sample_case_joined.unpersist()


# ==========================================
# BƯỚC 4: XỬ LÝ GEO EXPRESSION & METADATA
# ==========================================
logger.info("[Bước 4] GEO Expression & Metadata...")
raw_expr_text = spark.read.text(f"{RAW_DIR}/geo/expression/*.txt")
valid_expr_text = raw_expr_text.withColumn("cols_array", split(col("value"), "\t")).withColumn("col_size", size(col("cols_array"))).filter(col("col_size") > 2)
df_geo_expr = spark.read.option("header", "true").option("sep", "\t").csv(valid_expr_text.select("value").rdd.map(lambda r: r[0]))

if "ID_REF" in df_geo_expr.columns: df_geo_expr = df_geo_expr.withColumnRenamed("ID_REF", "Gene_Symbol")
df_geo_expr = df_geo_expr.filter(col("Gene_Symbol").isNotNull())
blacklist = ["gene_symbol", "id_ref", "identifier", "gene.title", "gene.symbol", "gene.id", "description", "gene_name", "hugo gene symbol"]
sample_cols = [c for c in df_geo_expr.columns if c.lower() not in blacklist and not c.startswith("_c")]

if len(sample_cols) > 0:
    expr_array = array(*[struct(lit(c).alias("Patient_ID"), when(col(c).rlike(numeric_regex), col(c).cast("float")).otherwise(lit(None).cast("float")).alias("Expression_Value")) for c in sample_cols])
    df_geo_long = df_geo_expr.select("Gene_Symbol", explode(expr_array).alias("expr")).select("Gene_Symbol", "expr.Patient_ID", "expr.Expression_Value").dropna(subset=["Expression_Value"])
    df_geo_long = df_geo_long.groupBy("Gene_Symbol", "Patient_ID").agg(avg("Expression_Value").alias("Expression_Value"))
    df_geo_long.persist(StorageLevel.MEMORY_AND_DISK)
    df_geo_long.repartition(16).write.mode("overwrite").parquet(f"{REFINED_DIR}/geo_expr_long.parquet")

raw_text_df = spark.read.text(f"{RAW_DIR}/geo/metadata/*")
split_df = raw_text_df.filter(~col("value").startswith("!") & ~col("value").startswith("^")).withColumn("cols_array", split(col("value"), "\t")).withColumn("col_size", size(col("cols_array")))
mode_size = split_df.groupBy("col_size").count().orderBy(col("count").desc()).first()["col_size"]
df_geo_meta = spark.read.option("header", "true").option("sep", "\t").csv(split_df.filter(col("col_size") == mode_size).select("value").rdd.map(lambda r: r[0]))

for c in ["Age", "StageNumeric", "AgeGT70", "OS_Time", "DSS_Time", "OSDSS_Time", "OSDSS60_Time", "DifferentiationNumeric", "scmodc20_sim"]:
    if c in df_geo_meta.columns: df_geo_meta = df_geo_meta.withColumn(c, when(col(c).rlike(numeric_regex), col(c).cast(DoubleType())).otherwise(lit(None).cast(DoubleType())))
for c in ["Stage_I_ADENO", "DEATH_before_5yrs", "DEATH_after_5yrs"]:
    if c in df_geo_meta.columns: df_geo_meta = df_geo_meta.withColumn(c, when(upper(col(c)) == "TRUE", 1).otherwise(0))

df_geo_meta.persist(StorageLevel.MEMORY_AND_DISK)
df_geo_meta.coalesce(1).write.mode("overwrite").parquet(f"{REFINED_DIR}/geo_meta_clean.parquet")

logger.info("[Bước 5] GEO Annotate...")
df_geo_meta_join = df_geo_meta.withColumnRenamed("Array", "Patient_ID")
df_geo_annotate = df_geo_long.join(broadcast(df_geo_meta_join), on="Patient_ID", how="inner")
df_geo_annotate.repartition(16).write.mode("overwrite").parquet(f"{REFINED_DIR}/geo/annotate.parquet")

df_geo_long.unpersist()
df_geo_meta.unpersist()


# ==========================================
# BƯỚC 6: XỬ LÝ STRING (GENE MAP)
# ==========================================
logger.info("[Bước 6] STRING - Bảng Gene Map...")
# [VÁ LỖI TẠI ĐÂY]: Đọc file csv trước, sau đó đổi tên cột có chứa dấu '#' rồi mới select
alias_df = spark.read.option("header", "true").option("sep", "\t").csv(f"{RAW_DIR}/STRING/aliases/*/*.txt.gz")
if "#string_protein_id" in alias_df.columns:
    alias_df = alias_df.withColumnRenamed("#string_protein_id", "protein_id")
alias_df = alias_df.select(col("protein_id"), col("alias"), col("source"))
alias_df = alias_df.withColumn("ensp_id", split(col("protein_id"), r"\.")[1])

ensg_df = alias_df.filter(col("source") == "Ensembl_gene").select("protein_id", col("alias").alias("gene_id"))
hgnc_df = alias_df.filter(col("source") == "Ensembl_HGNC_symbol").select("protein_id", col("alias").alias("gene_name"))
ensg_df = ensg_df.withColumn("rn", row_number().over(Window.partitionBy("protein_id").orderBy("gene_id"))).filter(col("rn") == 1).drop("rn")
hgnc_df = hgnc_df.withColumn("rn", row_number().over(Window.partitionBy("protein_id").orderBy("gene_name"))).filter(col("rn") == 1).drop("rn")

gene_map = alias_df.select("protein_id", "ensp_id").distinct().join(ensg_df, on="protein_id", how="left").join(hgnc_df, on="protein_id", how="left")
gene_map = gene_map.withColumn("gene_name_norm", upper(trim(col("gene_name"))))
gene_map = gene_map.withColumn("gene_confidence", when(col("gene_id").isNotNull() & col("gene_name").isNotNull(), "high").when(col("gene_id").isNotNull() | col("gene_name").isNotNull(), "medium").otherwise("low"))

gene_map.persist(StorageLevel.MEMORY_AND_DISK)
gene_map.repartition(4).write.mode("overwrite").parquet(f"{REFINED_DIR}/STRING/gene_map.parquet")


# ==========================================
# BƯỚC 7: XỬ LÝ STRING (EDGES PROTEIN & GENE)
# ==========================================
logger.info("[Bước 7] STRING - Bảng Edges...")
# [VÁ LỖI TẠI ĐÂY]: Đổi tên nếu các cột trong file có chứa dấu '#'
links_df = spark.read.option("header", "true").option("sep", " ").csv(f"{RAW_DIR}/STRING/links/*/*.txt.gz")
if "node1" in links_df.columns:
    links_df = links_df.withColumnRenamed("node1", "protein1")
if "node2" in links_df.columns:
    links_df = links_df.withColumnRenamed("node2", "protein2")

edges_protein = links_df.select(
    col("protein1").alias("protein_id_src"), col("protein2").alias("protein_id_dst"),
    col("combined_score").cast(DoubleType()).alias("combined_score_protein")
).withColumn("ensp_id_src", split(col("protein_id_src"), r"\.")[1]).withColumn("ensp_id_dst", split(col("protein_id_dst"), r"\.")[1]).withColumn("edge_weight_protein", col("combined_score_protein") / 1000.0)

edges_protein.persist(StorageLevel.MEMORY_AND_DISK)
edges_protein.repartition(16).write.mode("overwrite").parquet(f"{REFINED_DIR}/STRING/edges_protein.parquet")

edge_g_raw = edges_protein \
    .join(gene_map.select(col("protein_id").alias("protein_id_src"), col("gene_id").alias("raw_g_id_src"), col("gene_name_norm").alias("raw_g_name_src")), on="protein_id_src", how="inner") \
    .join(gene_map.select(col("protein_id").alias("protein_id_dst"), col("gene_id").alias("raw_g_id_dst"), col("gene_name_norm").alias("raw_g_name_dst")), on="protein_id_dst", how="inner")

edge_g_raw = edge_g_raw.withColumn("is_swapped", col("raw_g_name_src") > col("raw_g_name_dst"))
edge_g_raw = edge_g_raw \
    .withColumn("gene_name_src", when(col("is_swapped"), col("raw_g_name_dst")).otherwise(col("raw_g_name_src"))) \
    .withColumn("gene_name_dst", when(col("is_swapped"), col("raw_g_name_src")).otherwise(col("raw_g_name_dst"))) \
    .withColumn("gene_id_src", when(col("is_swapped"), col("raw_g_id_dst")).otherwise(col("raw_g_id_src"))) \
    .withColumn("gene_id_dst", when(col("is_swapped"), col("raw_g_id_src")).otherwise(col("raw_g_id_dst"))) \
    .withColumn("ord_ensp_id_src", when(col("is_swapped"), col("ensp_id_dst")).otherwise(col("ensp_id_src"))) \
    .withColumn("ord_ensp_id_dst", when(col("is_swapped"), col("ensp_id_src")).otherwise(col("ensp_id_dst")))

edges_gene = edge_g_raw.groupBy("gene_name_src", "gene_name_dst", "gene_id_src", "gene_id_dst").agg(
    max("combined_score_protein").alias("max_combined_score_gene"),
    max("edge_weight_protein").alias("max_edge_weight_gene"), count("*").alias("protein_edge_count"),
    max(struct(col("combined_score_protein"), col("ord_ensp_id_src"), col("ord_ensp_id_dst"))).alias("best_pair")
)
edges_gene = edges_gene.withColumn("best_ensp_id_src", col("best_pair.ord_ensp_id_src")).withColumn("best_ensp_id_dst", col("best_pair.ord_ensp_id_dst")).drop("best_pair")

edges_gene.persist(StorageLevel.MEMORY_AND_DISK)
edges_gene.repartition(8).write.mode("overwrite").parquet(f"{REFINED_DIR}/STRING/edges_gene.parquet")


# ==========================================
# BƯỚC 8: XỬ LÝ STRING (NODES PROTEIN & GENE)
# ==========================================
logger.info("[Bước 8] STRING - Bảng Nodes...")
p_nodes_src = edges_protein.select(col("protein_id_src").alias("protein_id"), col("edge_weight_protein"))
p_nodes_dst = edges_protein.select(col("protein_id_dst").alias("protein_id"), col("edge_weight_protein"))
p_deg = p_nodes_src.unionAll(p_nodes_dst).groupBy("protein_id").agg(count("*").alias("degree_protein"), sum("edge_weight_protein").alias("weighted_degree_protein"))

nodes_protein = p_deg.join(gene_map, on="protein_id", how="left").withColumn("protein_name", col("gene_name"))
nodes_protein.repartition(4).write.mode("overwrite").parquet(f"{REFINED_DIR}/STRING/nodes_protein.parquet")

g_nodes_src = edges_gene.select(col("gene_name_src").alias("gene_name_norm"), col("gene_id_src").alias("gene_id"), col("max_edge_weight_gene"))
g_nodes_dst = edges_gene.select(col("gene_name_dst").alias("gene_name_norm"), col("gene_id_dst").alias("gene_id"), col("max_edge_weight_gene"))
g_deg = g_nodes_src.unionAll(g_nodes_dst).groupBy("gene_name_norm", "gene_id").agg(count("*").alias("degree_gene"), sum("max_edge_weight_gene").alias("weighted_degree_gene"))

g_map_agg = gene_map.filter(col("gene_name_norm").isNotNull()).groupBy("gene_name_norm", "gene_id").agg(
    first("gene_name").alias("gene_name"), countDistinct("protein_id").alias("protein_count"), collect_set("protein_id").alias("protein_ids")
)
nodes_gene = g_deg.join(g_map_agg, on=["gene_name_norm", "gene_id"], how="inner")
nodes_gene.repartition(4).write.mode("overwrite").parquet(f"{REFINED_DIR}/STRING/nodes_gene.parquet")

logger.info("======= HOÀN TẤT TOÀN BỘ PIPELINE TRÊN COLAB =======")
spark.stop()

In [ ]:
import time
import os
import glob
from pyspark.sql.functions import col, count, when
# ==========================================
# CẤU HÌNH HỆ THỐNG COLAB & SPARK (AN TOÀN)
# ==========================================
from pyspark.sql import SparkSession

# Tắt phiên cũ nếu có (để giải phóng tài nguyên)
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("GDC_GEO_STRING_Colab_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "30") \
    .getOrCreate()

REFINED_DIR = "/content/drive/MyDrive/drugtarget_local/refined"

tables_to_check = {
    "1. GDC - Counts Clean": f"{REFINED_DIR}/gdc_counts_clean.parquet",
    "2. GDC - Sample Case": f"{REFINED_DIR}/gdc_sample_case_clean.parquet",
    "3. GDC - ANNOTATE": f"{REFINED_DIR}/gdc/annotate.parquet",
    "4. GEO - Expression Long": f"{REFINED_DIR}/geo_expr_long.parquet",
    "5. GEO - Metadata Clean": f"{REFINED_DIR}/geo_meta_clean.parquet",
    "6. GEO - ANNOTATE": f"{REFINED_DIR}/geo/annotate.parquet",
    "7. STRING - Gene Map": f"{REFINED_DIR}/STRING/gene_map.parquet",
    "8. STRING - Edges Gene": f"{REFINED_DIR}/STRING/edges_gene.parquet",
    "9. STRING - Nodes Gene": f"{REFINED_DIR}/STRING/nodes_gene.parquet"
}

for name, path in tables_to_check.items():
    print(f"\n{'='*80}")
    print(f"🚀 KIỂM TRA BẢNG: {name}")
    print(f"📂 PATH: {path}")
    print(f"{'='*80}")

    try:
        start = time.time()

        # Đọc parquet
        df = spark.read.parquet(path)
        df.cache()

        # Thống kê cơ bản
        row_count = df.count()
        col_count = len(df.columns)
        print(f"📊 Rows: {row_count:,} | Columns: {col_count}")

        # Schema
        print("\n🛠 SCHEMA:")
        df.printSchema()

        # Null check (10 cột đầu)
        print("\n🧪 NULL CHECK (10 cột đầu):")
        null_exprs = [count(when(col(c).isNull(), c)).alias(c) for c in df.columns[:10]]
        null_df = df.select(null_exprs)
        display(null_df) # Dùng display trực tiếp cho Spark DF

        # Sample 5 rows
        print("\n👁 SAMPLE 5 ROWS:")
        display(df.limit(5)) # Dùng display trực tiếp cho Spark DF

        # Partition check
        print(f"\n📦 PARTITIONS: {df.rdd.getNumPartitions()}")

        # [VÁ LỖI]: Thay dbutils bằng os.listdir để đếm file
        if os.path.exists(path):
            file_list = [f for f in os.listdir(path) if f.endswith(".parquet") or f.startswith("part-")]
            print(f"📁 Output parquet files: {len(file_list)}")

        end = time.time()
        print(f"\n⏱ Completed in {end-start:.2f} seconds")

        df.unpersist()

    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")


🚀 KIỂM TRA BẢNG: 1. GDC - Counts Clean
📂 PATH: /content/drive/MyDrive/drugtarget_local/refined/gdc_counts_clean.parquet
📊 Rows: 9,619,831 | Columns: 10

🛠 SCHEMA:
root
 |-- gene_id: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_type: string (nullable = true)
 |-- unstranded: long (nullable = true)
 |-- stranded_first: long (nullable = true)
 |-- stranded_second: long (nullable = true)
 |-- tpm_unstranded: double (nullable = true)
 |-- fpkm_unstranded: double (nullable = true)
 |-- fpkm_uq_unstranded: double (nullable = true)
 |-- file_id: string (nullable = true)


🧪 NULL CHECK (10 cột đầu):


DataFrame[gene_id: bigint, gene_name: bigint, gene_type: bigint, unstranded: bigint, stranded_first: bigint, stranded_second: bigint, tpm_unstranded: bigint, fpkm_unstranded: bigint, fpkm_uq_unstranded: bigint, file_id: bigint]


👁 SAMPLE 5 ROWS:


DataFrame[gene_id: string, gene_name: string, gene_type: string, unstranded: bigint, stranded_first: bigint, stranded_second: bigint, tpm_unstranded: double, fpkm_unstranded: double, fpkm_uq_unstranded: double, file_id: string]


📦 PARTITIONS: 3
📁 Output parquet files: 8

⏱ Completed in 60.28 seconds

🚀 KIỂM TRA BẢNG: 2. GDC - Sample Case
📂 PATH: /content/drive/MyDrive/drugtarget_local/refined/gdc_sample_case_clean.parquet
📊 Rows: 517 | Columns: 773

🛠 SCHEMA:
root
 |-- case_id: string (nullable = true)
 |-- demographic.ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- days_to_last_follow_up: double (nullable = true)
 |-- primary_diagnosis: string (nullable = true)
 |-- tissue_or_organ_of_origin: string (nullable = true)
 |-- diagnoses.1.days_to_last_follow_up: string (nullable = true)
 |-- diagnoses.1.primary_diagnosis: string (nullable = true)
 |-- diagnoses.1.tissue_or_organ_of_origin: string (nullable = true)
 |-- diagnoses.2.days_to_last_follow_up: string (nullable = true)
 |-- diagnoses.2.primary_diagnosis: string (nullable = true)
 |-- diagnoses.2.tissue_or_organ_of_origin: string (nullable = true)
 |-- diagnoses.3.days_to_last_follow_up: 

...


📦 PARTITIONS: 2
📁 Output parquet files: 8

⏱ Completed in 56.79 seconds

🚀 KIỂM TRA BẢNG: 9. STRING - Nodes Gene
📂 PATH: /content/drive/MyDrive/drugtarget_local/refined/STRING/nodes_gene.parquet
📊 Rows: 19,182 | Columns: 7

🛠 SCHEMA:
root
 |-- gene_name_norm: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- degree_gene: long (nullable = true)
 |-- weighted_degree_gene: double (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- protein_count: long (nullable = true)
 |-- protein_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)


🧪 NULL CHECK (10 cột đầu):


DataFrame[gene_name_norm: bigint, gene_id: bigint, degree_gene: bigint, weighted_degree_gene: bigint, gene_name: bigint, protein_count: bigint, protein_ids: bigint]


👁 SAMPLE 5 ROWS:


DataFrame[gene_name_norm: string, gene_id: string, degree_gene: bigint, weighted_degree_gene: double, gene_name: string, protein_count: bigint, protein_ids: array<string>]


📦 PARTITIONS: 2
📁 Output parquet files: 4

⏱ Completed in 0.70 seconds


In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit

# 1. Khởi tạo lại SparkSession an toàn (nếu đã có thì lấy lại)
spark = SparkSession.builder \
    .appName("Biomedical_Data_QA") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

REFINED_DIR = "/content/drive/MyDrive/drugtarget_local/refined"

# Danh sách bảng cần kiểm tra
tables_to_check = {
    "1. GDC Counts": f"{REFINED_DIR}/gdc_counts_clean.parquet",
    "2. GDC Sample Case": f"{REFINED_DIR}/gdc_sample_case_clean.parquet",
    "4. GEO Expr": f"{REFINED_DIR}/geo_expr_long.parquet",
    "5. GEO Meta": f"{REFINED_DIR}/geo_meta_clean.parquet",
    "7. STRING Gene Map": f"{REFINED_DIR}/STRING/gene_map.parquet"
}

# CHẠY QA CƠ BẢN
for name, path in tables_to_check.items():
    print(f"\n{'='*80}\n🚀 KIỂM TRA: {name}\n{'='*80}")
    df = spark.read.parquet(path)
    print(f"📊 Rows: {df.count():,}")
    print(f"🛠 Cột: {df.columns[:10]}...") # In 10 cột đầu

# ==========================================
# DEBUG JOINT (TÌM LỖI 0 DÒNG Ở GDC ANNOTATE)
# ==========================================
print(f"\n{'#'*80}\n🔍 BẮT ĐẦU DEBUG JOIN GDC\n{'#'*80}")

df_counts = spark.read.parquet(f"{REFINED_DIR}/gdc_counts_clean.parquet")
df_case = spark.read.parquet(f"{REFINED_DIR}/gdc_sample_case_clean.parquet")

# 1. Xem format file_id
print("\n--- Mẫu file_id từ Counts ---")
df_counts.select("file_id").distinct().show(5, truncate=False)

print("\n--- Mẫu file_id từ Sample Case ---")
df_case.select("file_id").distinct().show(5, truncate=False)

# 2. Kiểm tra giao điểm
count_counts = df_counts.select("file_id").distinct().count()
count_case = df_case.select("file_id").distinct().count()
intersect_count = df_counts.select("file_id").intersect(df_case.select("file_id")).count()

print(f"\n✅ Số ID Counts unique: {count_counts:,}")
print(f"✅ Số ID Sample Case unique: {count_case:,}")
print(f"🔥 SỐ ID KHỚP NHAU (JOIN SẼ CÓ SỐ DÒNG NÀY): {intersect_count:,}")

if intersect_count == 0:
    print("\n❌ LỖI: file_id không khớp! Có thể một bên có 'file_id=' ở đầu, bên kia thì không.")
    print("Gợi ý: Hãy thử dùng lệnh `split` hoặc `regexp_replace` để đồng bộ khóa.")

# ==========================================
# FIX LỖI NULL CHECK (Sử dụng backtick)
# ==========================================
print("\n🧪 NULL CHECK (Ví dụ trên bảng GDC Sample Case):")
df_debug = spark.read.parquet(f"{REFINED_DIR}/gdc_sample_case_clean.parquet")
# Dùng backtick để truy cập cột có dấu chấm
cols_to_check = ["`demographic.ethnicity`", "gender", "race", "file_id"]
for c in cols_to_check:
    try:
        null_count = df_debug.filter(col(c).isNull()).count()
        print(f"Cột {c} có {null_count} giá trị null")
    except:
        print(f"Cột {c} không tồn tại hoặc lỗi truy cập.")


🚀 KIỂM TRA: 1. GDC Counts
📊 Rows: 9,619,831
🛠 Cột: ['gene_id', 'gene_name', 'gene_type', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded', 'file_id']...

🚀 KIỂM TRA: 2. GDC Sample Case
📊 Rows: 517
🛠 Cột: ['case_id', 'demographic.ethnicity', 'gender', 'race', 'days_to_last_follow_up', 'primary_diagnosis', 'tissue_or_organ_of_origin', 'diagnoses.1.days_to_last_follow_up', 'diagnoses.1.primary_diagnosis', 'diagnoses.1.tissue_or_organ_of_origin']...

🚀 KIỂM TRA: 4. GEO Expr
📊 Rows: 11,026,590
🛠 Cột: ['Gene_Symbol', 'Patient_ID', 'Expression_Value']...

🚀 KIỂM TRA: 5. GEO Meta
📊 Rows: 1,106
🛠 Cột: ['Array', 'Age', 'Gender', 'Original Study', 'MergeGroup', 'Cohort', 'DSS_Status', 'DSS_Time', 'Differentiation_MD', 'Histology_progno']...

🚀 KIỂM TRA: 7. STRING Gene Map
📊 Rows: 19,699
🛠 Cột: ['protein_id', 'ensp_id', 'gene_id', 'gene_name', 'gene_name_norm', 'gene_confidence']...

######################################################

In [ ]:
import time
from pyspark.sql.functions import col, regexp_replace, count, when, lit
from pyspark.sql import SparkSession

# 1. Khởi tạo lại Session (không stop)
spark = SparkSession.builder.appName("Fix_GDC_Join").getOrCreate()
REFINED_DIR = "/content/drive/MyDrive/drugtarget_local/refined"

# ==========================================
# FIX LỖI JOIN GDC (GDC_ANNOTATE)
# ==========================================
print("🚀 Đang sửa lỗi Join GDC...")
df_counts = spark.read.parquet(f"{REFINED_DIR}/gdc_counts_clean.parquet")
df_sample = spark.read.parquet(f"{REFINED_DIR}/gdc_sample_case_clean.parquet")

# Xử lý prefix "file_id="
df_counts = df_counts.withColumn("file_id", regexp_replace(col("file_id"), "file_id=", ""))

# Join
df_gdc_annotate = df_counts.join(df_sample, on="file_id", how="inner")

# Ghi đè file
df_gdc_annotate.repartition(16).write.mode("overwrite").parquet(f"{REFINED_DIR}/gdc/annotate.parquet")
print(f"✅ Đã lưu xong GDC Annotate với {df_gdc_annotate.count():,} dòng!")

# ==========================================
# TEST LẠI VỚI CÚ PHÁP CỘT AN TOÀN (VÁ LỖI NULL CHECK)
# ==========================================
print("\n🧪 Đang chạy lại kiểm thử an toàn...")

# Danh sách bảng kiểm tra lại
final_check = {
    "GDC Annotate": f"{REFINED_DIR}/gdc/annotate.parquet"
}

for name, path in final_check.items():
    df = spark.read.parquet(path)

    # Kiểm tra số dòng (Phải > 0)
    print(f"\n📊 Bảng {name} có {df.count():,} dòng.")

    # VÁ LỖI TRUY CẬP CỘT: Sử dụng backtick ` ` bao quanh tên cột
    # Dùng list comprehension với format f"`{c}`"
    print("🧪 Null check (10 cột đầu):")
    null_exprs = [count(when(col(f"`{c}`").isNull(), c)).alias(c) for c in df.columns[:10]]
    null_df = df.select(null_exprs).toPandas()
    print(null_df)

🚀 Đang sửa lỗi Join GDC...
✅ Đã lưu xong GDC Annotate với 8,262,931 dòng!

🧪 Đang chạy lại kiểm thử an toàn...

📊 Bảng GDC Annotate có 8,262,931 dòng.
🧪 Null check (10 cột đầu):
   file_id  gene_id  gene_name  gene_type  unstranded  stranded_first  \
0        0        0          0          0           0               0   

   stranded_second  tpm_unstranded  fpkm_unstranded  fpkm_uq_unstranded  
0                0               0                0                   0  


In [ ]:
# ==========================================
# SCRIPT TỔNG DUYỆT DỮ LIỆU (QA/QC) TRÊN COLAB
# ==========================================
import time
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

# 1. Khởi tạo Spark Session an toàn (Không dùng spark.stop() để giữ session sống)
spark = SparkSession.builder \
    .appName("GDC_GEO_STRING_Colab_QA") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "30") \
    .getOrCreate()

REFINED_DIR = "/content/drive/MyDrive/drugtarget_local/refined"

# 2. Danh sách 9 bảng dữ liệu cốt lõi
tables_to_check = {
    "1. GDC - Counts Clean": f"{REFINED_DIR}/gdc_counts_clean.parquet",
    "2. GDC - Sample Case": f"{REFINED_DIR}/gdc_sample_case_clean.parquet",
    "3. GDC - ANNOTATE (Bảng ML)": f"{REFINED_DIR}/gdc/annotate.parquet",
    "4. GEO - Expression Long": f"{REFINED_DIR}/geo_expr_long.parquet",
    "5. GEO - Metadata Clean": f"{REFINED_DIR}/geo_meta_clean.parquet",
    "6. GEO - ANNOTATE": f"{REFINED_DIR}/geo/annotate.parquet",
    "7. STRING - Gene Map": f"{REFINED_DIR}/STRING/gene_map.parquet",
    "8. STRING - Edges Gene": f"{REFINED_DIR}/STRING/edges_gene.parquet",
    "9. STRING - Nodes Gene": f"{REFINED_DIR}/STRING/nodes_gene.parquet"
}

print("🚀 BẮT ĐẦU KIỂM THỬ TOÀN DIỆN PIPELINE...\n")

for name, path in tables_to_check.items():
    print(f"{'='*80}")
    print(f"📦 KIỂM TRA BẢNG: {name}")
    print(f"📂 PATH: {path}")
    print(f"{'='*80}")

    try:
        start = time.time()

        # Đọc dữ liệu từ Parquet
        df = spark.read.parquet(path)

        # Thống kê cơ bản
        row_count = df.count()
        col_count = len(df.columns)
        print(f"📊 Kích thước: {row_count:,} dòng | {col_count} cột")

        if row_count == 0:
            print("❌ CẢNH BÁO: Bảng không có dữ liệu (0 dòng)!")
            continue

        # In Schema (Chỉ in 5 cột đầu để đỡ rối mắt)
        print("\n🛠 SCHEMA (5 cột đầu):")
        for field in df.schema.fields[:5]:
            print(f"  - {field.name}: {field.dataType}")

        # Kiểm tra Null (10 cột đầu) - ĐÃ VÁ LỖI BACKTICK ĐỂ ĐỌC CỘT CÓ DẤU CHẤM
        print("\n🧪 NULL CHECK (10 cột đầu):")
        check_cols = df.columns[:10]
        null_exprs = [count(when(col(f"`{c}`").isNull(), c)).alias(c) for c in check_cols]
        null_df = df.select(null_exprs)
        display(null_df)

        # Xem trước 5 dòng đầu
        print("\n👁 SAMPLE 5 ROWS:")
        display(df.limit(5))

        # Đếm số lượng file vật lý sinh ra (Thay thế dbutils bằng os)
        if os.path.exists(path) and os.path.isdir(path):
            file_list = [f for f in os.listdir(path) if f.endswith(".parquet") or f.startswith("part-")]
            print(f"\n📁 Số lượng file vật lý: {len(file_list)}")

        # Partition hiện tại trên RAM
        print(f"🗂 Số lượng Partitions trên Spark: {df.rdd.getNumPartitions()}")

        end = time.time()
        print(f"⏱ Hoàn thành trong {end-start:.2f} giây\n")

    except Exception as e:
        print(f"\n❌ LỖI KHI ĐỌC BẢNG {name}: {str(e)}\n")

print("🎉 KẾT THÚC KIỂM THỬ!")

🚀 BẮT ĐẦU KIỂM THỬ TOÀN DIỆN PIPELINE...

📦 KIỂM TRA BẢNG: 1. GDC - Counts Clean
📂 PATH: /content/drive/MyDrive/drugtarget_local/refined/gdc_counts_clean.parquet
📊 Kích thước: 9,619,831 dòng | 10 cột

🛠 SCHEMA (5 cột đầu):
  - gene_id: StringType()
  - gene_name: StringType()
  - gene_type: StringType()
  - unstranded: LongType()
  - stranded_first: LongType()

🧪 NULL CHECK (10 cột đầu):


DataFrame[gene_id: bigint, gene_name: bigint, gene_type: bigint, unstranded: bigint, stranded_first: bigint, stranded_second: bigint, tpm_unstranded: bigint, fpkm_unstranded: bigint, fpkm_uq_unstranded: bigint, file_id: bigint]


👁 SAMPLE 5 ROWS:


DataFrame[gene_id: string, gene_name: string, gene_type: string, unstranded: bigint, stranded_first: bigint, stranded_second: bigint, tpm_unstranded: double, fpkm_unstranded: double, fpkm_uq_unstranded: double, file_id: string]


📁 Số lượng file vật lý: 8
🗂 Số lượng Partitions trên Spark: 3
⏱ Hoàn thành trong 0.54 giây

📦 KIỂM TRA BẢNG: 2. GDC - Sample Case
📂 PATH: /content/drive/MyDrive/drugtarget_local/refined/gdc_sample_case_clean.parquet
📊 Kích thước: 517 dòng | 773 cột

🛠 SCHEMA (5 cột đầu):
  - case_id: StringType()
  - demographic.ethnicity: StringType()
  - gender: StringType()
  - race: StringType()
  - days_to_last_follow_up: DoubleType()

🧪 NULL CHECK (10 cột đầu):


...


📁 Số lượng file vật lý: 8
🗂 Số lượng Partitions trên Spark: 2
⏱ Hoàn thành trong 0.58 giây

📦 KIỂM TRA BẢNG: 9. STRING - Nodes Gene
📂 PATH: /content/drive/MyDrive/drugtarget_local/refined/STRING/nodes_gene.parquet
📊 Kích thước: 19,182 dòng | 7 cột

🛠 SCHEMA (5 cột đầu):
  - gene_name_norm: StringType()
  - gene_id: StringType()
  - degree_gene: LongType()
  - weighted_degree_gene: DoubleType()
  - gene_name: StringType()

🧪 NULL CHECK (10 cột đầu):


DataFrame[gene_name_norm: bigint, gene_id: bigint, degree_gene: bigint, weighted_degree_gene: bigint, gene_name: bigint, protein_count: bigint, protein_ids: bigint]


👁 SAMPLE 5 ROWS:


DataFrame[gene_name_norm: string, gene_id: string, degree_gene: bigint, weighted_degree_gene: double, gene_name: string, protein_count: bigint, protein_ids: array<string>]


📁 Số lượng file vật lý: 4
🗂 Số lượng Partitions trên Spark: 2
⏱ Hoàn thành trong 0.34 giây

🎉 KẾT THÚC KIỂM THỬ!


In [ ]:
from pyspark.sql import SparkSession
import os

# Khởi tạo Spark nếu chưa có
spark = SparkSession.builder.getOrCreate()

REFINED_DIR = "/content/drive/MyDrive/drugtarget_local/refined"

# Danh sách các bảng chính cần xem
tables = [
    "gdc_counts_clean.parquet",
    "gdc_sample_case_clean.parquet",
    "gdc/annotate.parquet",
    "geo_expr_long.parquet",
    "geo_meta_clean.parquet",
    "geo/annotate.parquet",
    "STRING/gene_map.parquet",
    "STRING/edges_gene.parquet",
    "STRING/nodes_gene.parquet"
]

print("🔍 ĐANG TRÍCH XUẤT 10 DÒNG ĐẦU CỦA CÁC BẢNG ĐÃ CLEAN...\n")

for table_name in tables:
    path = os.path.join(REFINED_DIR, table_name)
    if os.path.exists(path):
        print("="*60)
        print(f"📄 BẢNG: {table_name}")
        print("="*60)
        try:
            df_preview = spark.read.parquet(path)
            display(df_preview.limit(10))
        except Exception as e:
            print(f"❌ Không thể đọc file {table_name}: {e}")
    else:
        print(f"⚠️ Không tìm thấy file: {table_name}")

print("\n✅ Đã hiển thị xong mẫu dữ liệu.")

🔍 ĐANG TRÍCH XUẤT 10 DÒNG ĐẦU CỦA CÁC BẢNG ĐÃ CLEAN...

📄 BẢNG: gdc_counts_clean.parquet


DataFrame[gene_id: string, gene_name: string, gene_type: string, unstranded: bigint, stranded_first: bigint, stranded_second: bigint, tpm_unstranded: double, fpkm_unstranded: double, fpkm_uq_unstranded: double, file_id: string]

📄 BẢNG: gdc_sample_case_clean.parquet


DataFrame[case_id: string, demographic.ethnicity: string, gender: string, race: string, days_to_last_follow_up: double, primary_diagnosis: string, tissue_or_organ_of_origin: string, diagnoses.1.days_to_last_follow_up: string, diagnoses.1.primary_diagnosis: string, diagnoses.1.tissue_or_organ_of_origin: string, diagnoses.2.days_to_last_follow_up: string, diagnoses.2.primary_diagnosis: string, diagnoses.2.tissue_or_organ_of_origin: string, diagnoses.3.days_to_last_follow_up: string, diagnoses.3.primary_diagnosis: string, diagnoses.3.tissue_or_organ_of_origin: string, diagnoses.4.days_to_last_follow_up: string, diagnoses.4.primary_diagnosis: string, diagnoses.4.tissue_or_organ_of_origin: string, diagnoses.5.primary_diagnosis: string, diagnoses.5.tissue_or_organ_of_origin: string, id: string, project_id: string, samples.0.portions.0.analytes.0.aliquots.0.aliquot_id: string, samples.0.portions.0.analytes.0.aliquots.0.submitter_id: string, samples.0.portions.0.analytes.0.aliquots.1.aliquot_i

📄 BẢNG: gdc/annotate.parquet


...

📄 BẢNG: STRING/edges_gene.parquet


DataFrame[gene_name_src: string, gene_name_dst: string, gene_id_src: string, gene_id_dst: string, max_combined_score_gene: double, max_edge_weight_gene: double, protein_edge_count: bigint, best_ensp_id_src: string, best_ensp_id_dst: string]

📄 BẢNG: STRING/nodes_gene.parquet


DataFrame[gene_name_norm: string, gene_id: string, degree_gene: bigint, weighted_degree_gene: double, gene_name: string, protein_count: bigint, protein_ids: array<string>]


✅ Đã hiển thị xong mẫu dữ liệu.


In [ ]:
# ==========================================
# BỔ SUNG: TẠO BẢNG GDC PROTEIN CODING & QC
# ==========================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct

spark = SparkSession.builder.getOrCreate()
REFINED_DIR = "/content/drive/MyDrive/drugtarget_local/refined"

print("🚀 Đang trích xuất bảng GDC Protein Coding...")

# 1. Đọc bảng GDC Annotate tổng hợp
df_gdc = spark.read.parquet(f"{REFINED_DIR}/gdc/annotate.parquet")

# 2. Lọc protein_coding như yêu cầu
df_gdc_protein = df_gdc.filter(col("gene_type") == "protein_coding")

# 3. Ghi bảng protein coding
df_gdc_protein.repartition(16).write.mode("overwrite").parquet(f"{REFINED_DIR}/gdc/counts_clean_protein_coding_long.parquet")

# 4. Tạo báo cáo QC nhanh (GDC)
print("\n📊 BÁO CÁO QC GDC:")
stats = df_gdc.select(
    count("*").alias("total_rows"),
    countDistinct("file_id").alias("distinct_files"),
    countDistinct("gene_id").alias("distinct_genes")
).collect()[0]

protein_stats = df_gdc_protein.select(
    count("*").alias("protein_rows"),
    countDistinct("gene_id").alias("distinct_protein_genes")
).collect()[0]

print(f"- Tổng số dòng (All Genes): {stats['total_rows']:,}")
print(f"- Số lượng File ID: {stats['distinct_files']}")
print(f"- Số lượng Gene (All): {stats['distinct_genes']}")
print(f"- Số dòng Protein Coding: {protein_stats['protein_rows']:,}")
print(f"- Số lượng Gene Protein Coding: {protein_stats['distinct_protein_genes']}")

print("\n✅ Đã hoàn thành bổ sung bảng Protein Coding cho GDC.")

🚀 Đang trích xuất bảng GDC Protein Coding...

📊 BÁO CÁO QC GDC:
- Tổng số dòng (All Genes): 8,262,931
- Số lượng File ID: 517
- Số lượng Gene (All): 19286
- Số dòng Protein Coding: 8,262,931
- Số lượng Gene Protein Coding: 19286

✅ Đã hoàn thành bổ sung bảng Protein Coding cho GDC.


### 📋 Tổng hợp thuộc tính các bảng dữ liệu sau khi Refined

Danh sách dưới đây mô tả cấu trúc các cột (schema) và kiểu dữ liệu của các bảng quan trọng nhất trong quy trình.

In [ ]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.getOrCreate()
REFINED_DIR = "/content/drive/MyDrive/drugtarget_local/refined"

tables = {
    "GDC - Counts Clean": "gdc_counts_clean.parquet",
    "GDC - Sample Case": "gdc_sample_case_clean.parquet",
    "GDC - Annotate (Long)": "gdc/annotate.parquet",
    "GEO - Expression Long": "geo_expr_long.parquet",
    "GEO - Metadata": "geo_meta_clean.parquet",
    "STRING - Gene Map": "STRING/gene_map.parquet",
    "STRING - Edges Gene": "STRING/edges_gene.parquet",
    "STRING - Nodes Gene": "STRING/nodes_gene.parquet"
}

for name, rel_path in tables.items():
    path = os.path.join(REFINED_DIR, rel_path)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"📌 BẢNG: {name}")
        print(f"📂 PATH: {rel_path}")
        print(f"{'='*60}")
        df = spark.read.parquet(path)
        df.printSchema()
    else:
        print(f"\n⚠️ Không tìm thấy bảng: {name} ({rel_path})")


📌 BẢNG: GDC - Counts Clean
📂 PATH: gdc_counts_clean.parquet
root
 |-- gene_id: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_type: string (nullable = true)
 |-- unstranded: long (nullable = true)
 |-- stranded_first: long (nullable = true)
 |-- stranded_second: long (nullable = true)
 |-- tpm_unstranded: double (nullable = true)
 |-- fpkm_unstranded: double (nullable = true)
 |-- fpkm_uq_unstranded: double (nullable = true)
 |-- file_id: string (nullable = true)


📌 BẢNG: GDC - Sample Case
📂 PATH: gdc_sample_case_clean.parquet
root
 |-- case_id: string (nullable = true)
 |-- demographic.ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- days_to_last_follow_up: double (nullable = true)
 |-- primary_diagnosis: string (nullable = true)
 |-- tissue_or_organ_of_origin: string (nullable = true)
 |-- diagnoses.1.days_to_last_follow_up: string (nullable = true)
 |-- diagnoses.1.primary_diagnosis: str

### 📊 Tóm tắt Thuộc tính các Bảng (Refined)

| Nhóm | Tên Bảng | Thuộc tính chính | Mục đích |
| :--- | :--- | :--- | :--- |
| **GDC** | `gdc_counts_clean` | `gene_id`, `gene_name`, `tpm_unstranded`, `file_id` | Biểu hiện gen (RNA-Seq) |
| | `gdc_sample_case_clean` | `case_id`, `primary_diagnosis`, `survival_time`, `sample_type` | Thông tin lâm sàng & sinh tồn |
| | `gdc/annotate` | *Hợp nhất Counts & Sample* | Bảng tổng hợp phục vụ Machine Learning |
| **GEO** | `geo_expr_long` | `Gene_Symbol`, `Patient_ID`, `Expression_Value` | Biểu hiện gen từ dữ liệu Microarray |
| | `geo_meta_clean` | `Array` (Patient_ID), `Age`, `Gender`, `Stage` | Thông tin bệnh nhi/mẫu bệnh phẩm |
| **STRING** | `STRING/gene_map` | `protein_id`, `ensp_id`, `gene_id`, `gene_name` | Ánh xạ giữa Protein và Gene |
| | `STRING/edges_gene` | `gene_name_src`, `gene_name_dst`, `max_edge_weight_gene` | Mạng lưới tương tác giữa các gen |
| | `STRING/nodes_gene` | `gene_name`, `degree_gene`, `weighted_degree_gene` | Đặc trưng mạng lưới (độ quan trọng của gen) |

### 🧬 Tóm tắt Thuộc tính nhóm Protein (STRING)

| Nhóm | Tên Bảng | Thuộc tính chính | Mục đích |
| :--- | :--- | :--- | :--- |
| **STRING (Protein)** | `STRING/edges_protein` | `protein_id_src`, `protein_id_dst`, `edge_weight_protein` | Mạng lưới tương tác gốc ở cấp độ Protein (ENSP) |
| | `STRING/nodes_protein` | `protein_id`, `degree_protein`, `weighted_degree_protein`, `protein_name` | Chỉ số mạng lưới và tên tương ứng của từng Protein |